In [ ]:
# === BOOTSTRAP: RUN THIS FIRST ===
import os, sys
from pathlib import Path
REPO_PATH = Path("/content/drive/MyDrive/MaintainAI/code")
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH))
print("ROOT:", REPO_PATH)
print("Exists:", REPO_PATH.exists())
print("src exists:", (REPO_PATH / "src").exists())

# 03 — Preprocessing + causal features
Raw immutable → processed parquet. Engine-level split; scaler fit on train engines only.

In [ ]:
import joblib, json
from pathlib import Path
from src.data_cmapss import load_fd001, add_rul, add_capped_rul, engine_split, assert_disjoint
from src.features import build_features, feature_columns, fit_scaler

# Load raw data
tr, te, rul = load_fd001('CMAPSSData')
tr = add_capped_rul(add_rul(tr), cap=125)

# Engine-level split (deterministic, seed=42)
tr_units, va_units = engine_split(sorted(int(u) for u in tr['unit'].unique()), 0.2, 42)
assert_disjoint(tr_units, va_units)
print(f'Train engines: {len(tr_units)} | Val engines: {len(va_units)}')

# Build causal features (shift(1) ensures no future leakage)
cols = feature_columns()
feat = build_features(tr)
print(f'Features built: {feat.shape}, feature columns: {len(cols)}')
print(f'NaN count in feature cols: {int(feat[cols].isna().sum().sum())}')

# Save processed features
Path('data/processed').mkdir(parents=True, exist_ok=True)
feat.to_parquet('data/processed/train_features.parquet', index=False)

# Fit scaler on TRAIN engines only (no val/test leakage)
scaler = fit_scaler(feat[feat['unit'].isin(tr_units)], cols)
Path('models/predictive').mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, 'models/predictive/scaler_fd001.joblib')

# Save feature schema for later use
schema = {
    'feature_columns': cols,
    'window': 30,
    'rul_cap': 125,
    'train_units': tr_units,
    'val_units': va_units
}
with open('models/predictive/feature_schema.json', 'w') as f:
    json.dump(schema, f, indent=1)

print('✓ Saved: data/processed/train_features.parquet')
print('✓ Saved: models/predictive/scaler_fd001.joblib')
print('✓ Saved: models/predictive/feature_schema.json')

In [ ]:
# Verify causal property: rollmean at cycle N uses only cycles < N
import pandas as pd
from src.data_cmapss import INFORMATIVE_SENSORS

# Create synthetic rising sensor to verify
df = pd.DataFrame({
    'unit': [1] * 5,
    'cycle': [1, 2, 3, 4, 5],
    **{s: ([10.0, 20.0, 30.0, 40.0, 50.0] if s == 's3' else [1.0] * 5) for s in INFORMATIVE_SENSORS},
})
feat_test = build_features(df, window=30)
# Row 3 (cycle 3): history = cycles 1-2 -> mean(10,20)=15
assert feat_test.loc[2, 's3_rollmean_30'] == 15.0, f"Expected 15.0, got {feat_test.loc[2, 's3_rollmean_30']}"
# First cycle: no history -> trend/delta zero
assert feat_test.loc[0, 's3_trend'] == 0.0
assert feat_test.loc[0, 's3_delta_base'] == 0.0
print('✓ Causal features verified: no future leakage')

In [ ]:
# Verify scaler statistics on train set
from src.features import apply_scaler
import numpy as np

tr_feat = feat[feat['unit'].isin(tr_units)]
scaled = apply_scaler(tr_feat, cols, scaler)
print(f'Scaled train mean: {scaled[cols].to_numpy().mean():.6f}')
print(f'Scaled train std: {scaled[cols].to_numpy().std():.6f}')
print(f'All finite: {np.isfinite(scaled[cols].to_numpy()).all()}')
print('✓ Scaler fitted on train engines only')